In [54]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

# 1. LOAD DATA
data = pd.read_csv("/kaggle/input/datafixxxx/dataset_training_final_FIXED.csv")

# Tambahkan query_id
data['query_id'] = data['query'].astype('category').cat.codes

# 2. PISAHKAN FITUR & LABEL (query_id TIDAK MASUK MODEL)
X = data.drop(columns=['label', 'query', 'text', 'len_q', 'len_t', 'query_id'])
y = data['label']
qid = data['query_id']

X_train, X_test, y_train, y_test, q_train, q_test = train_test_split(
    X, y, qid,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. LOAD MODEL
lr   = joblib.load('/kaggle/input/singlemodelfix/logisticregression (3).pkl')
rf   = joblib.load('/kaggle/input/modelfixxtanpasvm/randomforest (2).pkl')
xgb  = joblib.load('/kaggle/input/modelfixxtanpasvm/xgboost (2).pkl')
lgbm = joblib.load('/kaggle/input/modelfixxtanpasvm/lightgbm (2).pkl')
svm = joblib.load('/kaggle/input/svmfixx/svm (3).pkl')

models = {
    "RandomForest": rf,
    "XGBoost": xgb,
    "LightGBM": lgbm,
    "LogisticRegression": lr,
    "SVM": svm
}

# 4. DEFINISI METRIK MRR & RECALL
def mrr(y_true, y_score):
    order = np.argsort(y_score)[::-1]
    y_sorted = y_true[order]
    hits = np.where(y_sorted > 0)[0]
    return 1.0 / (hits[0] + 1) if len(hits) > 0 else 0.0

def recall_at_k(y_true, y_score, k=10):
    total_rel = np.sum(y_true > 0)
    if total_rel == 0:
        return 0.0
    top_k = np.argsort(y_score)[-k:][::-1]
    rel_in_top_k = np.sum(y_true[top_k] > 0)
    return rel_in_top_k / total_rel

# 5. GROUP DATA PER QUERY
unique_q = np.unique(q_test)

X_groups = [X_test[q_test == q] for q in unique_q]
y_groups = [y_test[q_test == q].values for q in unique_q]

# 6. EVALUASI SETIAP MODEL
def dcg_at_k(relevances, k):
    relevances = np.asfarray(relevances)[:k]
    return np.sum((2**relevances - 1) / np.log2(np.arange(2, len(relevances) + 2)))

def ndcg_at_k(y_true, y_score, k):
    order = np.argsort(y_score)[::-1]
    y_true_sorted = y_true[order]

    dcg = dcg_at_k(y_true_sorted, k)

    # Ideal DCG
    ideal_sorted = np.sort(y_true)[::-1]
    idcg = dcg_at_k(ideal_sorted, k)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def evaluate_model(model, k=10):
    y_score_groups = [model.predict_proba(Xg)[:, 1] for Xg in X_groups]

    ndcgs = []
    mrrs = []
    recalls = []

    for y_true, y_score in zip(y_groups, y_score_groups):

        # Jika tidak ada yang relevan → skip
        if np.sum(y_true) == 0:
            continue

        # nDCG custom (aman)
        ndcgs.append(ndcg_at_k(y_true, y_score, k))

        # MRR
        mrrs.append(mrr(y_true, y_score))

        # Recall@K
        recalls.append(recall_at_k(y_true, y_score, k))

    return {
        f"nDCG@{k}": np.mean(ndcgs),
        "MRR": np.mean(mrrs),
        f"Recall@{k}": np.mean(recalls)
    }

# 7. HASIL
for name, model in models.items():
    print(f"\n=== Model: {name} ===")
    result = evaluate_model(model, k=3)
    for m, v in result.items():
        print(f"{m}: {v:.4f}")


=== Model: RandomForest ===
nDCG@3: 0.9642
MRR: 0.9521
Recall@3: 0.9991

=== Model: XGBoost ===
nDCG@3: 0.9648
MRR: 0.9531
Recall@3: 0.9991

=== Model: LightGBM ===
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[LightGBM] [Warning] Unknown parameter: gamma
[Light

In [53]:
from sklearn.metrics import recall_score

def evaluate_recall(model):
    # Prediksi label biner (0/1)
    y_pred = model.predict(X_test)

    # Hitung recall keseluruhan
    return recall_score(y_test, y_pred)

for name, model in models.items():
    print(f"{name} Recall: {evaluate_recall(model):.4f}")


RandomForest Recall: 0.4246
XGBoost Recall: 0.4322
[LightGBM] [Warning] Unknown parameter: gamma
LightGBM Recall: 0.4302
LogisticRegression Recall: 0.3867
SVM Recall: 0.2994
